**`deploy_pipeline`**

Review and deploy an orchestrated pipeline run (Snakemake) on a cluster or locally.

Starting from a terminal recipe and a geography, this notebook derives every
(stage, recipe, admin unit) job from the recipe tree (`RecipeDAG`), displays the
dependency graph, shows which jobs would run and why, and — after the dry-run
inspection gate — deploys the workflow (SGE via `workflow/profiles/scc`, or
locally with `--executor local`).

Pass `--deploy` to submit; without it, the run stops at the inspection gate.

Workflow: `workflow/Snakefile`

# Configure

In [ ]:
import argparse
import getpass
import shutil

import pandas as pd
from IPython.display import Markdown, display

from openplaces.config import cfg
from openplaces.flow import RecipeDAG, run_subprocess
from openplaces.flow.dag import STAGES, rule_name
from openplaces.flow.submit import deploy, dry_run

In [ ]:
parser = argparse.ArgumentParser(
    description='Review and deploy an orchestrated pipeline run (Snakemake)'
)
parser.add_argument(
    '--recipe_id',
    help='Terminal recipe to build (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to process (e.g. "US-NC-BS")',
    nargs='*',
)
parser.add_argument(
    '--executor',
    help='Where to run: "scc" submits via qsub (workflow/profiles/scc), '
    '"local" runs on this machine',
    choices=['scc', 'local'],
    default='scc',
)
parser.add_argument(
    '--cores',
    help='Parallel cores for --executor local',
    type=int,
    default=4,
)
parser.add_argument(
    '--forcerun',
    help='Recipe IDs whose jobs are forced to re-run, cascading to their '
    'successors (snakemake --forcerun)',
    nargs='*',
)
parser.add_argument(
    '--email',
    help='Notification address for the workflow onsuccess/onerror handlers',
)
parser.add_argument(
    '--deploy',
    help='Actually submit the jobs; without it the run stops at the '
    'inspection gate after the dry run',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US_footprint-cheer-2026 '
    '--admin_ids US-NC-CE '  # Carteret (pilot)
    # '--admin_ids US-NC-BS '  # Brunswick
    # '--admin_ids US-NC-BS US-NC-CE US-NC-ON '  # 3-county validation
    '--executor local '
    # '--executor scc '
    # '--forcerun US_footprint-spine-2026 '
    # '--deploy '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

# Build the DAG

One job = one (stage, recipe, admin unit) node, derived from the recipe tree.
Recipes saving finer than the requested units (e.g. per-town image caches)
expand into child-unit jobs; this requires the admin boundaries to be
ingested (see the warning otherwise).

In [ ]:
dag = RecipeDAG(args.recipe_id, admin_ids=args.admin_ids)

jobs = pd.DataFrame(dag.nodes())
print(f'{len(jobs)} jobs for {len(args.admin_ids or [])} admin unit(s):\n')
print(
    jobs['stage'].value_counts().reindex(list(STAGES)).dropna().astype(int).to_string()
)
jobs

# Visualize the DAG

Stage colors: ingest (blue), harmonize (green), enrich (yellow), curate (red).
Graphs larger than ~30 jobs auto-collapse to one node per recipe, labeled
with the admin-unit count.

In [ ]:
mermaid = dag.to_mermaid()

# Renders in VS Code notebooks (and mermaid-enabled JupyterLab); if your
# frontend shows raw text instead, print(mermaid) and paste into mermaid.live
display(Markdown(f'```mermaid\n{mermaid}\n```'))

# Keep the source next to the SGE logs for the record
mermaid_path = cfg.get_dir('logs') / 'sge' / 'dag.mmd'
mermaid_path.parent.mkdir(parents=True, exist_ok=True)
mermaid_path.write_text(mermaid, encoding='utf-8')
print(f'Mermaid source: {mermaid_path}')
# print(mermaid)

# Review scheduled updates

Library-side preview of which jobs would run and why (one file stat per
job; reasons: output missing, inputs newer than output, or an upstream job
is scheduled). The authoritative decision is Snakemake's dry run at the
inspection gate below.

In [ ]:
plan = dag.plan()

n_run = int(plan['will_run'].sum())
print(f'{n_run} of {len(plan)} jobs would run.\n')
print(plan.loc[plan['will_run'], 'reason'].value_counts().to_string())

# Overview grid: one row per recipe, one column per admin unit
overview = plan.assign(
    stage=pd.Categorical(plan['stage'], list(STAGES)),
    admin_id=plan['admin_id'].fillna('(global)'),
    marker=plan['will_run'].map({True: 'run', False: ''}),
).pivot_table(
    index=['stage', 'recipe_id'],
    columns='admin_id',
    values='marker',
    aggfunc='first',
    fill_value='-',
    observed=True,
)
overview

In [ ]:
# Full per-job detail (outputs, sizes, reasons)
# plan

# Map the selected admin units (requires ingested admin boundaries):
# import openplaces as op
# counties = op.get_admin('US', 3, geom=True)
# ax = counties.plot(color='none', edgecolor='grey', linewidth=0.2)
# counties[counties.index.isin(args.admin_ids)].plot(ax=ax, color='red')

# Inspection gate

Snakemake's dry run is the authoritative schedule; it is printed here and
stored next to the SGE logs for the post-hoc record. **Never submit jobs
without inspecting them first**: the next cell stops the run unless
`--deploy` was passed.

In [ ]:
workflow_config = {
    'recipe': args.recipe_id,
    'admin_ids': args.admin_ids,
}
if args.email:
    workflow_config['email'] = args.email

# Translate forced recipe IDs into the generated per-job rule names
forcerun_rules = [
    rule_name(node) for node in dag.nodes() if node.recipe_id in (args.forcerun or [])
]
forcerun_args = ('--forcerun', *forcerun_rules) if forcerun_rules else ()

returncode, output, dryrun_path = dry_run(
    config=workflow_config, extra_args=forcerun_args, verbose=True
)
if returncode != 0:
    raise SystemExit('Dry run failed; fix the issues above before deploying.')

In [ ]:
# Never submit jobs without inspecting them first
if not args.deploy:
    raise SystemExit('Inspect the dry run above, then re-run with --deploy to submit.')

# Deploy

SCC: one qsub job per DAG node via the cluster profile (200-job throttle,
per-stage memory with attempt scaling, retries). Local: parallel processes
on this machine. Either way, Snakemake deletes `until_consumed` outputs
(cache parquets, image caches) once every consuming job has finished, and
re-dispatched complete jobs no-op.

In [ ]:
if args.executor == 'scc':
    returncode = deploy(
        profile='workflow/profiles/scc',
        config=workflow_config,
        extra_args=forcerun_args,
    )
else:
    returncode = deploy(
        cores=args.cores,
        config=workflow_config,
        extra_args=forcerun_args,
    )
print(f'snakemake finished with exit code {returncode}')

# Monitor

In [ ]:
# Jobs currently in the cluster queue (SGE only; skipped elsewhere)
if shutil.which('qstat'):
    run_subprocess(['qstat', '-u', getpass.getuser()], verbose=True)
else:
    print('qstat not available on this machine.')

In [ ]:
# Re-runnable: share of jobs whose outputs are now up to date
progress = dag.plan()
n_done = int((~progress['will_run']).sum())
print(f'{n_done} of {len(progress)} jobs up to date.')
progress[progress['will_run']][['stage', 'recipe_id', 'admin_id', 'reason']]

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# test_script(*args_list, committed=COMMIT)